In [13]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datetime import datetime

import models
import data_loader
import base
import meta
import plot
import evaluation
from trainers import ANN, LSTM, MetaModel

In [ ]:
TICKER      = "AAPL"      # Change to any ticker you want
START_DATE  = "2013-01-01"
END_DATE    = datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
WINDOW_SIZE = 1          # How many past days the model sees
BATCH_SIZE  = 16
N_SPLITS    = 5           # TimeSeriesSplit folds; we use the last one
EPOCHS = 500


OUTPUT_DIR  = "outputs"
EVALUATION_DIR = "evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EVALUATION_DIR, exist_ok=True)

### Data Loader

In [15]:
# Downloading and building features from data
df = data_loader.download_data(TICKER, START_DATE, END_DATE)
data = data_loader.build_features(df)
n_features = data.shape[1]
scaled, f_scaler, c_scaler = data_loader.normalize(data)

# Creatong sequences and splitting dataset for training
X, y = data_loader.create_sequences(scaled, WINDOW_SIZE)
X_train, X_test, y_train, y_test = data_loader.split_data(X, y)

# Splitting into train and test datasets
train_loader = DataLoader(
    models.StockDataset(X_train, y_train), 
    batch_size=BATCH_SIZE, shuffle=True
    )
test_loader  = DataLoader(
    models.StockDataset(X_test,  y_test), 
    batch_size=BATCH_SIZE, shuffle=False
    )

[*********************100%***********************]  1 of 1 completed

Downloaded 3369 rows for AAPL
Train: 2807 samples | Test: 561 samples


### Models Loader

In [16]:
lstm_model = LSTM(n_features=n_features)
ann_model  = ANN(n_features=n_features, seq_len=WINDOW_SIZE)
meta_model = MetaModel()

params = (sum(p.numel() for p in lstm_model.parameters()) +
                sum(p.numel() for p in ann_model.parameters()) +
                sum(p.numel() for p in meta_model.parameters()))

In [17]:
params

130405

### Training MetaModel Process

In [18]:
# LSTM and ANN model training
lstm_model, lstm_tr_loss, lstm_val_loss = base.train(
    lstm_model, train_loader, test_loader, name="LSTM", epochs=EPOCHS
)
ann_model, ann_tr_loss, ann_val_loss = base.train(
    ann_model, train_loader, test_loader, name="ANN", epochs=EPOCHS
)

# Get predictions from each model
lstm_tr_preds, ann_tr_preds, y_tr_meta = base.predict(
    lstm_model, ann_model, train_loader
)
lstm_val_preds, ann_val_preds, y_val_meta = base.predict(
    lstm_model, ann_model, test_loader
)


Training LSTM on cpu
  Epoch  10/500 | Train MSE: 0.000396 | Val MSE: 0.000432
  Epoch  20/500 | Train MSE: 0.000288 | Val MSE: 0.000243
  Epoch  30/500 | Train MSE: 0.000265 | Val MSE: 0.000391
  Epoch  40/500 | Train MSE: 0.000250 | Val MSE: 0.000351
  Epoch  50/500 | Train MSE: 0.000234 | Val MSE: 0.000456
  Epoch  60/500 | Train MSE: 0.000239 | Val MSE: 0.000303
  Epoch  70/500 | Train MSE: 0.000238 | Val MSE: 0.000408
  Epoch  80/500 | Train MSE: 0.000234 | Val MSE: 0.000412
  Epoch  90/500 | Train MSE: 0.000242 | Val MSE: 0.000405
  Epoch 100/500 | Train MSE: 0.000229 | Val MSE: 0.000387
  Epoch 110/500 | Train MSE: 0.000223 | Val MSE: 0.000393
  Epoch 120/500 | Train MSE: 0.000253 | Val MSE: 0.000391
  Epoch 130/500 | Train MSE: 0.000233 | Val MSE: 0.000393
  Epoch 140/500 | Train MSE: 0.000224 | Val MSE: 0.000392
  Epoch 150/500 | Train MSE: 0.000238 | Val MSE: 0.000393
  Epoch 160/500 | Train MSE: 0.000244 | Val MSE: 0.000393
  Epoch 170/500 | Train MSE: 0.000230 | Val MSE: 0

In [19]:
# Using previous LSTM and ANN predicted results to train the meta-model
meta_model, meta_tr_loss, meta_val_loss = meta.train(
        meta_model,
        lstm_tr_preds, ann_tr_preds, y_tr_meta,
        lstm_val_preds, ann_val_preds, y_val_meta,
        epochs=EPOCHS
    )

Training Meta-Model (Stacking Layer)
  Epoch  10/500 | Train MSE: 0.188537 | Val MSE: 0.601954
  Epoch  20/500 | Train MSE: 0.018239 | Val MSE: 0.080807
  Epoch  30/500 | Train MSE: 0.001055 | Val MSE: 0.010789
  Epoch  40/500 | Train MSE: 0.000266 | Val MSE: 0.003447
  Epoch  50/500 | Train MSE: 0.000139 | Val MSE: 0.001967
  Epoch  60/500 | Train MSE: 0.000082 | Val MSE: 0.001285
  Epoch  70/500 | Train MSE: 0.000063 | Val MSE: 0.000941
  Epoch  80/500 | Train MSE: 0.000058 | Val MSE: 0.000784
  Epoch  90/500 | Train MSE: 0.000057 | Val MSE: 0.000722
  Epoch 100/500 | Train MSE: 0.000057 | Val MSE: 0.000703
  Epoch 110/500 | Train MSE: 0.000057 | Val MSE: 0.000694
  Epoch 120/500 | Train MSE: 0.000057 | Val MSE: 0.000695
  Epoch 130/500 | Train MSE: 0.000057 | Val MSE: 0.000689
  Epoch 140/500 | Train MSE: 0.000057 | Val MSE: 0.000683
  Epoch 150/500 | Train MSE: 0.000057 | Val MSE: 0.000686
  Epoch 160/500 | Train MSE: 0.000057 | Val MSE: 0.000698
  Epoch 170/500 | Train MSE: 0.0000

### Evaluating all models

In [20]:
# Get scaled predictions
lstm_pred_s, y_true_s = evaluation.predict(lstm_model, test_loader)
ann_pred_s, _ = evaluation.predict(ann_model,  test_loader)
ens_pred_s, _ = evaluation.ensemble_predictions(
    lstm_model, ann_model, meta_model, test_loader
    )

In [21]:
# Inverse-transform to real $ values
y_true_r    = evaluation.inverse_scale(y_true_s,   c_scaler)
lstm_pred_r = evaluation.inverse_scale(lstm_pred_s, c_scaler)
ann_pred_r  = evaluation.inverse_scale(ann_pred_s,  c_scaler)
ens_pred_r  = evaluation.inverse_scale(ens_pred_s,  c_scaler)

In [22]:
# Compute and display metrics (real $ scale)
print("\n  ─── Test Set Performance (real $ scale) ───")
print(f"  {'Model':<25} {'R²':>8}  {'MAE':>8}  {'MSE':>10}  {'RMSE':>8}")
results = {}
results["LSTM"]     = evaluation.compute_metrics(y_true_r, lstm_pred_r, "LSTM")
results["ANN"]      = evaluation.compute_metrics(y_true_r, ann_pred_r,  "ANN")
results["LSTM+ANN"] = evaluation.compute_metrics(y_true_r, ens_pred_r,  "LSTM+ANN (Ensemble)")


  ─── Test Set Performance (real $ scale) ───
  Model                           R²       MAE         MSE      RMSE
  LSTM                      R²=+0.9806  MAE=3.2020  MSE=19.5776  RMSE=4.4247
  ANN                       R²=+0.9372  MAE=6.2755  MSE=63.3762  RMSE=7.9609
  LSTM+ANN (Ensemble)       R²=+0.9688  MAE=4.3329  MSE=31.4902  RMSE=5.6116


In [23]:
# Improvement summary
r2_ann = results["ANN"]["R2"]
r2_ens = results["LSTM+ANN"]["R2"]
if r2_ann > 0:
    improvement = (r2_ens - r2_ann) / abs(r2_ann) * 100
    print(f"\n  Ensemble R² improvement over ANN: {improvement:+.1f}%")


  Ensemble R² improvement over ANN: +3.4%


### Saving plots

In [24]:
plot.training_curves(
    {"LSTM": (lstm_tr_loss, lstm_val_loss),
        "ANN":  (ann_tr_loss,  ann_val_loss),
        "Meta": (meta_tr_loss, meta_val_loss)},
    save_path=f"{OUTPUT_DIR}/training_curves.png"
)
plot.predictions(
    y_true_r,
    {"LSTM": lstm_pred_r, "ANN": ann_pred_r, "LSTM+ANN": ens_pred_r},
    ticker=TICKER,
    save_path=f"{OUTPUT_DIR}/predictions.png"
)
plot.metrict_comparison(
    results,
    save_path=f"{OUTPUT_DIR}/metrics_comparison.png"
)

# Save model weights
torch.save(lstm_model.state_dict(), f"{OUTPUT_DIR}/lstm_weights.pt")
torch.save(ann_model.state_dict(),  f"{OUTPUT_DIR}/ann_weights.pt")
torch.save(meta_model.state_dict(), f"{OUTPUT_DIR}/meta_weights.pt")

print(f"  Weights saved to {OUTPUT_DIR}/")

print("\n" + "="*60)
print("  Training complete.")
print(f"  Best model: LSTM+ANN Ensemble")
print(f"  R² = {r2_ens:.4f} | RMSE = {results['LSTM+ANN']['RMSE']:.4f}")
print("="*60 + "\n")

  Saved: outputs/training_curves.png
  Saved: outputs/predictions.png
  Saved: outputs/metrics_comparison.png
  Weights saved to outputs/

  Training complete.
  Best model: LSTM+ANN Ensemble
  R² = 0.9688 | RMSE = 5.6116

